## Bentley auth and Evo discovery using direct API calls

If you prefer to use the Evo Python SDK instead, check out the `sdk-examples.ipynb` notebook.

This notebook shows the two main authentication flows using direct API calls with common Python packages:

- Native, SPA, or Web apps that use the authorization code flow.
- Service apps that use the client credentials flow.

### Native, SPA, or Web app auth

Create a **native**, **SPA**, or **web app** in the [Seequent Developer Portal](https://developer.seequent.com/my-apps). This gives you the required `client ID` and `redirect URL` parameters that you need in the next cells to start the sign-in flow. 

Web apps also require a `client secret` parameter.
NOTE: Delete the `client secret` from this notebook before committing it to source control or sharing it with others.

In [ ]:
# Enter your client ID and redirect URL.
client_id = "your-client-id"
redirect_url = "your-redirect-url"

# Enter your client secret if you are using a Web app. If you are using a Native or SPA app, leave this empty.
client_secret = ""

# The scope list below is preloaded with the standard Evo scopes typically required
# for these examples. If you also need a refresh token, add "offline_access" to the list.
scope = ["evo.workspace", "evo.discovery", "evo.object", "evo.blocksync", "evo.file"]

base_uri = "https://ims.bentley.com"
authorization_url = f"{base_uri}/connect/authorize"
access_token_url = f"{base_uri}/connect/token"

In [ ]:
import datetime
import webbrowser
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse

import jwt
from authlib.common.security import generate_token
from authlib.integrations.requests_client import OAuth2Session


def get_access_token(
    client_id: str,
    client_secret: str | None,
    redirect_url: str,
    authorization_url: str,
    access_token_url: str,
    scope: list[str],
) -> str:
    """Get an access token."""

    port = urlparse(redirect_url).port
    if port is None:
        raise ValueError(
            "redirect_url must include an explicit port so the local callback "
            "server can start, for example: http://localhost:8080/callback"
        )

    class OAuthHttpServer(HTTPServer):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.authorization_code = ""

    class OAuthHttpHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            self.send_response(200)
            self.send_header("Content-Type", "text/html")
            self.end_headers()

            client.fetch_token(
                url=access_token_url,
                authorization_response=self.path,
                state=state,
                code_verifier=code_verifier,
                grant_type="authorization_code",
            )

            self.wfile.write(
                """
<html lang="en">
  <head>
    <title>Seequent Evo - Authorization successful</title>
    <style>
      body {
        font-family: sans-serif;
        background: #eaeaea;
        color: #1a1a1a;
        margin: 0;
        min-height: 100vh;
        display: flex;
        align-items: center;
        justify-content: center;
      }
      .container {
        background: #f3f3f3;
        padding: 80px;
        text-align: center;
      }
    </style>
  </head>
  <body>
    <div class="container">
      <svg width="84" height="84" viewBox="0 0 84 84" fill="none" xmlns="http://www.w3.org/2000/svg">
        <path d="M22.5159 49.8778C21.7023 49.0642 21.7023 47.7451 22.5159 46.9315L25.4621 43.9852C26.2757 43.1716 27.5948 43.1716 28.4084 43.9852L32.8278 48.4046L54.9249 26.3075C55.7385 25.4939 57.0576 25.4939 57.8712 26.3075L60.8175 29.2538C61.6311 30.0674 61.6311 31.3865 60.8175 32.2001L34.301 58.7166C33.4874 59.5302 32.1683 59.5302 31.3547 58.7166L22.5159 49.8778Z" fill="#477B5E"/>
        <path fill-rule="evenodd" clip-rule="evenodd" d="M83.3333 41.6667C83.3333 64.6785 64.6785 83.3333 41.6667 83.3333C18.6548 83.3333 0 64.6785 0 41.6667C0 18.6548 18.6548 0 41.6667 0C64.6785 0 83.3333 18.6548 83.3333 41.6667ZM75 41.6667C75 60.0762 60.0762 75 41.6667 75C23.2572 75 8.33333 60.0762 8.33333 41.6667C8.33333 23.2572 23.2572 8.33333 41.6667 8.33333C60.0762 8.33333 75 23.2572 75 41.6667Z" fill="#477B5E"/>
      </svg>
      <h1>Authorization successful!</h1>
      <p>You have successfully authenticated with Seequent Evo.<br><br>You may now close this window and return to your terminal or application.</p>
    </div>
    <script>setTimeout("window.close()", 2500);</script>
  </body>
</html>
""".encode("UTF-8")
            )

    with OAuthHttpServer(("", port), OAuthHttpHandler) as httpd:
        # Web apps use client_secret_post; public clients default to client_secret_basic.
        token_endpoint_auth_method = "client_secret_post" if client_secret else "client_secret_basic"
        client = OAuth2Session(
            client_id=client_id,
            client_secret=client_secret,
            scope=scope,
            redirect_uri=redirect_url,
            code_challenge_method="S256",
            token_endpoint_auth_method=token_endpoint_auth_method,
        )

        code_verifier = generate_token(48)
        auth_uri, state = client.create_authorization_url(authorization_url, code_verifier=code_verifier)
        webbrowser.open_new(auth_uri)
        httpd.handle_request()

    return client.token["access_token"]


desktop_access_token = get_access_token(
    client_id=client_id,
    client_secret=client_secret,
    redirect_url=redirect_url,
    authorization_url=authorization_url,
    access_token_url=access_token_url,
    scope=scope,
)

print("Access token:")
print(desktop_access_token)

decoded = jwt.decode(desktop_access_token, options={"verify_signature": False}, algorithms=["RS256"])
exp_timestamp = decoded["exp"]
exp_datetime = datetime.datetime.fromtimestamp(exp_timestamp, datetime.timezone.utc)
print(f"Token expires at:\n{exp_datetime} UTC")

### Service app auth

Create a **service app** in the [Seequent Developer Portal](https://developer.seequent.com/my-apps). This gives you the `client ID` and `client secret` that you need in the next cell to start the sign-in flow.

NOTE: Delete the `client secret` from this notebook before committing it to source control or sharing it with others.

In [ ]:
import datetime
from http import HTTPStatus

import jwt
import requests

# Enter your client ID, client secret and Evo scopes.
client_id = "your-service-app-client-id"
client_secret = "your-service-app-client-secret"

# The scope list below is preloaded with the standard Evo scopes typically required
# for these examples.
scope = "evo.workspace evo.discovery evo.object evo.blocksync evo.file"

# Enter the token endpoint URL.
url = "https://ims.bentley.com/connect/token"

# Enter the parameters for the token request.
params = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": scope,
}

# Make the token request and print the access token.
response = requests.post(url, data=params)
if response.status_code != HTTPStatus.OK:
    raise RuntimeError(f"Failed to get token: {response.status_code} {response.reason}")

service_access_token = response.json()["access_token"]
print("Access token:")
print(service_access_token)

# Decode the access token to get the expiration time.
# Note: In a production application, you should verify the token signature.
decoded = jwt.decode(service_access_token, options={"verify_signature": False}, algorithms=["RS256"])
exp_timestamp = decoded["exp"]

exp_datetime = datetime.datetime.fromtimestamp(exp_timestamp, datetime.timezone.utc)
print(f"Token expires at: {exp_datetime} UTC")

### Evo Discovery API

Evo Discovery is the service that tells your application where Evo capabilities are available for the current authenticated environment. Instead of hard-coding service endpoints, you can call Discovery to find the correct URLs for the Evo services your app wants to use.

In practice, this is useful when your application needs to work out which endpoint to call for services such as block models, geoscience objects, or files.

Use an access token from either auth flow above.

In [ ]:
import json

# Choose which token to use for the Discovery API call.
access_token = desktop_access_token  # or service_access_token

# Prepare the authorization header.
auth_header = {"Authorization": "Bearer " + access_token}

# Define the Evo services to be searched.
services = ["blockmodel", "geoscienceobject", "file"]
params = {"service": services}

# Define the Evo Discovery API URL.
base_url = "https://discover.api.seequent.com/evo/identity/v2/discovery"

# Make the GET request to the Evo Discovery API.
response = requests.get(base_url, headers=auth_header, params=params)
response_output = json.dumps(response.json(), indent=4)
print(f"GET {base_url}?{response.url.split('?')[1]}")
print(response_output)